In [9]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable

print("[INIT] Gold notebook started.")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 11, Finished, Available, Finished, False)

[INIT] Gold notebook started.


In [10]:
# ============================================================
# CELL 2 — PARAMETERS
# ============================================================

run_id = "MANUAL_TEST"

print(f"[PARAMETERS] Run ID: {run_id}")

if run_id is None or str(run_id).strip() == "":
    raise ValueError("Required parameter 'run_id' is missing.")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 12, Finished, Available, Finished, False)

[PARAMETERS] Run ID: MANUAL_TEST


In [11]:
# ============================================================
# CELL 3 — READ SILVER
# ============================================================

silver_df = spark.table("silver_matches")
print(f"[GOLD] Silver rows read: {silver_df.count()}")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 13, Finished, Available, Finished, False)

[GOLD] Silver rows read: 187


In [12]:
# ============================================================
# CELL 4 — DIM_DATE (static, pre-generated, 2020-2030)
# ============================================================

if not spark.catalog.tableExists("dim_date"):

    print("[GOLD] Building dim_date (2020-2030)...")

    date_df = spark.sql("""
        SELECT explode(sequence(
            to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day
        )) AS full_date
    """).withColumn(
        "date_sk", F.date_format("full_date", "yyyyMMdd").cast("int")
    ).withColumn(
        "year", F.year("full_date")
    ).withColumn(
        "month", F.month("full_date")
    ).withColumn(
        "day", F.dayofmonth("full_date")
    ).withColumn(
        "day_name", F.date_format("full_date", "EEEE")
    ).select("date_sk", "full_date", "year", "month", "day", "day_name")

    date_df.write.format("delta").mode("overwrite").saveAsTable("dim_date")
    print(f"[GOLD] dim_date created: {date_df.count()} rows.")

else:
    print("[GOLD] dim_date already exists. Skipping.")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 14, Finished, Available, Finished, False)

[GOLD] dim_date already exists. Skipping.


In [13]:
# # ============================================================
# # CELL 5 — DIM_SERIES (simple upsert, no history needed)
# # ============================================================

# series_source = silver_df.select(
#     F.col("series_id"), F.trim(F.col("series_name")).alias("series_name")
# ).filter(F.col("series_id").isNotNull()).dropDuplicates(["series_id"])

# if not spark.catalog.tableExists("dim_series"):

#     series_df = series_source.withColumn(
#         "series_sk", F.monotonically_increasing_id()
#     )
#     series_df.write.format("delta").mode("overwrite").saveAsTable("dim_series")

# else:

#     target = DeltaTable.forName(spark, "dim_series")
#     target.alias("t").merge(
#         series_source.alias("s"), "t.series_id = s.series_id"
#     ).whenMatchedUpdate(set={"series_name": "s.series_name"}) \
#      .whenNotMatchedInsert(values={
#          "series_sk": F.monotonically_increasing_id(),
#          "series_id": "s.series_id",
#          "series_name": "s.series_name"
#      }).execute()

# print(f"[GOLD] dim_series rows: {spark.table('dim_series').count()}")

# ============================================================
# CELL 5 — DIM_SERIES (simple upsert, no history needed)
# ============================================================

series_source = silver_df.select(
    F.col("series_id"), F.trim(F.col("series_name")).alias("series_name")
).filter(F.col("series_id").isNotNull()).dropDuplicates(["series_id"])

if not spark.catalog.tableExists("dim_series"):

    series_df = series_source.withColumn("series_sk", F.monotonically_increasing_id())
    series_df.write.format("delta").mode("overwrite").saveAsTable("dim_series")

else:

    target = DeltaTable.forName(spark, "dim_series")
    existing_ids = target.toDF().select("series_id")

    new_rows = series_source.join(existing_ids, on="series_id", how="left_anti")
    max_sk = target.toDF().agg(F.max("series_sk")).collect()[0][0] or 0

    if new_rows.count() > 0:
        new_rows_with_sk = new_rows.rdd.zipWithIndex().map(
            lambda x: (x[0]["series_id"], x[0]["series_name"], max_sk + 1 + x[1])
        ).toDF(["series_id", "series_name", "series_sk"])

        new_rows_with_sk.write.format("delta").mode("append").saveAsTable("dim_series")
        print(f"[GOLD] Inserted {new_rows.count()} new series.")

    # Update names for existing series that changed
    target.alias("t").merge(
        series_source.alias("s"), "t.series_id = s.series_id"
    ).whenMatchedUpdate(set={"series_name": "s.series_name"}).execute()

print(f"[GOLD] dim_series rows: {spark.table('dim_series').count()}")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 15, Finished, Available, Finished, False)

[GOLD] dim_series rows: 28


In [14]:
# # ============================================================
# # CELL 6 — DIM_VENUE (simple upsert)
# # ============================================================

# venue_source = silver_df.select(
#     F.trim(F.col("venue_ground")).alias("venue_ground"),
#     F.trim(F.col("venue_city")).alias("venue_city")
# ).filter(F.col("venue_ground").isNotNull()).dropDuplicates(["venue_ground", "venue_city"])

# if not spark.catalog.tableExists("dim_venue"):

#     venue_df = venue_source.withColumn("venue_sk", F.monotonically_increasing_id())
#     venue_df.write.format("delta").mode("overwrite").saveAsTable("dim_venue")

# else:

#     target = DeltaTable.forName(spark, "dim_venue")
#     target.alias("t").merge(
#         venue_source.alias("s"),
#         "t.venue_ground = s.venue_ground AND t.venue_city = s.venue_city"
#     ).whenNotMatchedInsert(values={
#         "venue_sk": F.monotonically_increasing_id(),
#         "venue_ground": "s.venue_ground",
#         "venue_city": "s.venue_city"
#     }).execute()

# print(f"[GOLD] dim_venue rows: {spark.table('dim_venue').count()}")


# ============================================================
# CELL 6 — DIM_VENUE (simple upsert)
# ============================================================

venue_source = silver_df.select(
    F.trim(F.col("venue_ground")).alias("venue_ground"),
    F.trim(F.col("venue_city")).alias("venue_city")
).filter(F.col("venue_ground").isNotNull()).dropDuplicates(["venue_ground", "venue_city"])

if not spark.catalog.tableExists("dim_venue"):

    venue_df = venue_source.withColumn("venue_sk", F.monotonically_increasing_id())
    venue_df.write.format("delta").mode("overwrite").saveAsTable("dim_venue")

else:

    target = DeltaTable.forName(spark, "dim_venue")
    existing = target.toDF().select("venue_ground", "venue_city")

    new_rows = venue_source.join(existing, on=["venue_ground", "venue_city"], how="left_anti")
    max_sk = target.toDF().agg(F.max("venue_sk")).collect()[0][0] or 0

    if new_rows.count() > 0:
        new_rows_with_sk = new_rows.rdd.zipWithIndex().map(
            lambda x: (x[0]["venue_ground"], x[0]["venue_city"], max_sk + 1 + x[1])
        ).toDF(["venue_ground", "venue_city", "venue_sk"])

        new_rows_with_sk.write.format("delta").mode("append").saveAsTable("dim_venue")
        print(f"[GOLD] Inserted {new_rows.count()} new venues.")

print(f"[GOLD] dim_venue rows: {spark.table('dim_venue').count()}")


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 16, Finished, Available, Finished, False)

[GOLD] dim_venue rows: 50


In [15]:
# ============================================================
# CELL 7 — DIM_TEAM (SCD Type 2)
# ============================================================
#
# Change detection on: team_name, team_sname
# On change -> expire old row (is_current=false, effective_to),
# insert new row (is_current=true, effective_from=today)
# ============================================================

team1_source = silver_df.select(
    F.col("team1_id").alias("team_id"),
    F.trim(F.col("team1_name")).alias("team_name"),
    F.trim(F.col("team1_sname")).alias("team_sname")
)

team2_source = silver_df.select(
    F.col("team2_id").alias("team_id"),
    F.trim(F.col("team2_name")).alias("team_name"),
    F.trim(F.col("team2_sname")).alias("team_sname")
)

teams_source = team1_source.unionByName(team2_source) \
    .filter(F.col("team_id").isNotNull()) \
    .dropDuplicates(["team_id"])

print(f"[GOLD] Distinct teams in this run: {teams_source.count()}")

if not spark.catalog.tableExists("dim_team"):

    print("[GOLD] dim_team does not exist. Creating initial version...")

    dim_team_df = teams_source \
        .withColumn("team_sk", F.monotonically_increasing_id()) \
        .withColumn("effective_from", F.current_date()) \
        .withColumn("effective_to", F.lit(None).cast("date")) \
        .withColumn("is_current", F.lit(True))

    dim_team_df.write.format("delta").mode("overwrite").saveAsTable("dim_team")
    print(f"[GOLD] dim_team created: {dim_team_df.count()} rows.")

else:

    dim_team = DeltaTable.forName(spark, "dim_team")
    current_df = dim_team.toDF().filter("is_current = true")

    # Rows that are NEW teams or CHANGED attributes vs current version
    changes = teams_source.alias("s").join(
        current_df.alias("c"), on="team_id", how="left"
    ).filter(
        F.col("c.team_id").isNull()
        | (F.coalesce(F.col("c.team_name"), F.lit("")) != F.coalesce(F.col("s.team_name"), F.lit("")))
        | (F.coalesce(F.col("c.team_sname"), F.lit("")) != F.coalesce(F.col("s.team_sname"), F.lit("")))
    ).select("s.team_id", "s.team_name", "s.team_sname")

    change_count = changes.count()
    print(f"[GOLD] New/changed teams detected: {change_count}")

    if change_count > 0:

        # Step 1: expire current rows for changed team_ids
        dim_team.alias("t").merge(
            changes.alias("s"),
            "t.team_id = s.team_id AND t.is_current = true"
        ).whenMatchedUpdate(set={
            "is_current": "false",
            "effective_to": "current_date()"
        }).execute()

        # Step 2: insert new current versions
        max_sk = dim_team.toDF().agg(F.max("team_sk")).collect()[0][0] or 0

        new_rows = changes.rdd.zipWithIndex().map(
            lambda x: (*x[0], max_sk + 1 + x[1])
        ).toDF(["team_id", "team_name", "team_sname", "team_sk"]) \
            .withColumn("effective_from", F.current_date()) \
            .withColumn("effective_to", F.lit(None).cast("date")) \
            .withColumn("is_current", F.lit(True))

        new_rows.write.format("delta").mode("append").saveAsTable("dim_team")

        print(f"[GOLD] Inserted {change_count} new dim_team version(s).")

    else:
        print("[GOLD] No team changes detected. dim_team unchanged.")

print(f"[GOLD] dim_team total rows: {spark.table('dim_team').count()}")



StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 17, Finished, Available, Finished, False)

[GOLD] Distinct teams in this run: 137
[GOLD] New/changed teams detected: 0
[GOLD] No team changes detected. dim_team unchanged.
[GOLD] dim_team total rows: 137


In [16]:
# ============================================================
# CELL 8 — FACT_MATCH
# ============================================================

from pyspark.sql import functions as F
from delta.tables import DeltaTable


# ------------------------------------------------------------
# 1. Load dimension tables
# ------------------------------------------------------------

dim_team_current = (
    spark.table("dim_team")
    .filter(F.col("is_current") == True)
)

dim_series_lookup = spark.table("dim_series")

dim_venue_lookup = spark.table("dim_venue")


# ------------------------------------------------------------
# 2. Build FACT_MATCH dataframe
# ------------------------------------------------------------

fact_df = (
    silver_df

    # Team 1 lookup
    .join(
        dim_team_current.select(
            F.col("team_id").alias("t1_id"),
            F.col("team_sk").alias("team1_sk")
        ),
        silver_df.team1_id == F.col("t1_id"),
        "left"
    )

    # Team 2 lookup
    .join(
        dim_team_current.select(
            F.col("team_id").alias("t2_id"),
            F.col("team_sk").alias("team2_sk")
        ),
        silver_df.team2_id == F.col("t2_id"),
        "left"
    )

    # Series lookup
    .join(
        dim_series_lookup.select(
            "series_id",
            "series_sk"
        ),
        on="series_id",
        how="left"
    )

    # Venue lookup
    .join(
        dim_venue_lookup,
        on=["venue_ground", "venue_city"],
        how="left"
    )

    # Date surrogate key
    .withColumn(
        "match_date_sk",
        F.date_format(
            F.col("start_date"),
            "yyyyMMdd"
        ).cast("int")
    )

    # Select fact columns
    .select(
        F.col("match_id"),
        F.col("series_sk"),
        F.col("venue_sk"),
        F.col("team1_sk"),
        F.col("team2_sk"),
        F.col("match_date_sk"),
        F.col("match_type"),
        F.col("match_format"),
        F.col("status"),
        F.col("state"),
        F.col("source_type"),
        F.col("team1_runs"),
        F.col("team1_wickets"),
        F.col("team1_overs"),
        F.col("team2_runs"),
        F.col("team2_wickets"),
        F.col("team2_overs")
    )

    # Gold load timestamp
    .withColumn(
        "gold_updated_at",
        F.current_timestamp()
    )
)


# ------------------------------------------------------------
# 3. Fact table name
# ------------------------------------------------------------

fact_table = "fact_match"


# ------------------------------------------------------------
# 4. Create table if it doesn't exist
# ------------------------------------------------------------

if not spark.catalog.tableExists(fact_table):

    print("[GOLD] fact_match does not exist. Creating table...")

    fact_with_sk = (
        fact_df
        .withColumn(
            "match_sk",
            F.monotonically_increasing_id()
        )
        .select(
            "match_sk",
            "match_id",
            "series_sk",
            "venue_sk",
            "team1_sk",
            "team2_sk",
            "match_date_sk",
            "match_type",
            "match_format",
            "status",
            "state",
            "source_type",
            "team1_runs",
            "team1_wickets",
            "team1_overs",
            "team2_runs",
            "team2_wickets",
            "team2_overs",
            "gold_updated_at"
        )
    )

    fact_with_sk.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(fact_table)

    print("[GOLD] fact_match created successfully.")


# ------------------------------------------------------------
# 5. MERGE if table already exists
# ------------------------------------------------------------

else:

    print("[GOLD] fact_match already exists. Performing MERGE...")

    target = DeltaTable.forName(
        spark,
        fact_table
    )

    # Add surrogate key to source
    fact_df_with_sk = (
        fact_df
        .withColumn(
            "match_sk",
            F.monotonically_increasing_id()
        )
    )

    (
        target.alias("t")

        .merge(
            fact_df_with_sk.alias("s"),
            "t.match_id = s.match_id"
        )

        # ----------------------------------------------------
        # Existing match
        # Do NOT update match_sk
        # ----------------------------------------------------

        .whenMatchedUpdate(
            set={
                "series_sk": "s.series_sk",
                "venue_sk": "s.venue_sk",
                "team1_sk": "s.team1_sk",
                "team2_sk": "s.team2_sk",
                "match_date_sk": "s.match_date_sk",

                "match_type": "s.match_type",
                "match_format": "s.match_format",

                "status": "s.status",
                "state": "s.state",
                "source_type": "s.source_type",

                "team1_runs": "s.team1_runs",
                "team1_wickets": "s.team1_wickets",
                "team1_overs": "s.team1_overs",

                "team2_runs": "s.team2_runs",
                "team2_wickets": "s.team2_wickets",
                "team2_overs": "s.team2_overs",

                "gold_updated_at": "s.gold_updated_at"
            }
        )

        # ----------------------------------------------------
        # New match
        # Insert match_sk + all other columns
        # ----------------------------------------------------

        .whenNotMatchedInsert(
            values={
                "match_sk": "s.match_sk",
                "match_id": "s.match_id",

                "series_sk": "s.series_sk",
                "venue_sk": "s.venue_sk",
                "team1_sk": "s.team1_sk",
                "team2_sk": "s.team2_sk",
                "match_date_sk": "s.match_date_sk",

                "match_type": "s.match_type",
                "match_format": "s.match_format",

                "status": "s.status",
                "state": "s.state",
                "source_type": "s.source_type",

                "team1_runs": "s.team1_runs",
                "team1_wickets": "s.team1_wickets",
                "team1_overs": "s.team1_overs",

                "team2_runs": "s.team2_runs",
                "team2_wickets": "s.team2_wickets",
                "team2_overs": "s.team2_overs",

                "gold_updated_at": "s.gold_updated_at"
            }
        )

        .execute()
    )

    print("[GOLD] fact_match MERGE completed.")


# ------------------------------------------------------------
# 6. Final row count
# ------------------------------------------------------------

print(
    f"[GOLD] fact_match rows: "
    f"{spark.table(fact_table).count()}"
)

StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 18, Finished, Available, Finished, False)

[GOLD] fact_match already exists. Performing MERGE...
[GOLD] fact_match MERGE completed.
[GOLD] fact_match rows: 187


In [17]:
# ============================================================
# CELL 9 — FINAL VALIDATION
# ============================================================

print("=" * 60)
print("GOLD LAYER COMPLETED")
print("=" * 60)
print(f"dim_date    : {spark.table('dim_date').count()} rows")
print(f"dim_series  : {spark.table('dim_series').count()} rows")
print(f"dim_venue   : {spark.table('dim_venue').count()} rows")
print(f"dim_team    : {spark.table('dim_team').count()} rows (current: {spark.table('dim_team').filter('is_current = true').count()})")
print(f"fact_match  : {spark.table('fact_match').count()} rows")
print(f"Run ID      : {run_id}")
print("Status      : SUCCESS")
print("=" * 60)


StatementMeta(, 3cb20e63-1e09-499d-9b9d-4031ed054957, 19, Finished, Available, Finished, False)

GOLD LAYER COMPLETED
dim_date    : 4018 rows
dim_series  : 28 rows
dim_venue   : 50 rows
dim_team    : 137 rows (current: 137)
fact_match  : 187 rows
Run ID      : MANUAL_TEST
Status      : SUCCESS
